# Phase 2 finale — the routing pass

**Runtime: L4 GPU** (Runtime → Change runtime type → L4). ~1–1.5 h total, ~5 units.
**Checkpointing built in:** progress saves to Drive every 25 bugs — a disconnect
costs minutes, not the run. Re-running resumes where it left off.

**What this does (Amendment A2):** the base Qwen model attempts every TRAIN-split
bug **8 times at temperature 1.0** — the exact exploration style GRPO will use.
Each bug is then routed by outcome:
- **0/8 solved → SFT pile** (the model can't do these; worked solutions must teach them)
- **1–7/8 → RL pile** (mixed outcomes = learning signal for try-and-grade)
- **8/8 → easy pile** (nothing to learn; excluded from RL)

This same data IS the GRPO variance-gate pre-flight — we're buying two deliverables
with one GPU hour.

In [ ]:
# Setup: Drive, fresh repo, canonical prompt module, the bug corpus
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, importlib
PHASE2 = '/content/drive/MyDrive/rl-post-training/phase2'
os.makedirs(PHASE2, exist_ok=True)
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
for p in ('/content/ptl/src',):
    if p not in sys.path:
        sys.path.insert(0, p)
for mod in ('prompts',):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
from prompts import build_repair_prompt, extract_code
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
bugs = json.load(open('/content/ptl/data/data_v0_bugs.json'))
train_bugs = [b for b in bugs if b['split'] == 'train']
print(len(bugs), 'bugs total |', len(train_bugs), 'train-split bugs to route')

In [ ]:
# Load the base model (bf16 on the L4)
import torch
assert torch.cuda.is_bf16_supported(), 'Switch runtime to L4 (Runtime > Change runtime type).'
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL, padding_side='left')
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map='cuda')
model.eval()
print('Model loaded:', MODEL)

In [ ]:
# Generation helpers: k=8 attempts per bug at temperature 1.0 (GRPO's regime)
K = 8
PROMPTS_PER_BATCH = 4   # 4 prompts x 8 attempts = 32 sequences per forward pass

def chat(prompt_text):
    return tok.apply_chat_template([{'role': 'user', 'content': prompt_text}],
                                   tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def attempt_batch(batch):
    texts = [chat(build_repair_prompt(b['buggy'], b['test_list'])) for b in batch]
    enc = tok(texts, return_tensors='pt', padding=True).to('cuda')
    out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=0.95,
                         num_return_sequences=K, max_new_tokens=512,
                         pad_token_id=tok.pad_token_id)
    gen = out[:, enc['input_ids'].shape[1]:]
    decoded = tok.batch_decode(gen, skip_special_tokens=True)
    return [ [extract_code(t) for t in decoded[i*K:(i+1)*K]] for i in range(len(batch)) ]

In [ ]:
# SMOKE: 2 bugs, eyeball the extracted attempts before the long run
sample_out = attempt_batch(train_bugs[:2])
for b, attempts in zip(train_bugs[:2], sample_out):
    print('=' * 60)
    print(b['id'], '|', b['description'])
    print('--- attempt 1 (extracted) ---')
    print(attempts[0][:600])
print('\nHealthy = complete function definitions. If you see prose/empty text, STOP and report.')

In [ ]:
# FULL RUN with checkpoint-resume (saves to Drive every 25 bugs)
import time
CKPT = f'{PHASE2}/routing_samples_ckpt.json'
samples = json.load(open(CKPT)) if os.path.exists(CKPT) else {}
todo = [b for b in train_bugs if b['id'] not in samples]
print(f'resuming: {len(samples)} done, {len(todo)} to go')
t0 = time.time()
for i in range(0, len(todo), PROMPTS_PER_BATCH):
    batch = todo[i:i+PROMPTS_PER_BATCH]
    for b, attempts in zip(batch, attempt_batch(batch)):
        samples[b['id']] = attempts
    if (i // PROMPTS_PER_BATCH) % 6 == 5 or i + PROMPTS_PER_BATCH >= len(todo):
        json.dump(samples, open(CKPT, 'w'))
        done = len(samples); rate = (done - (len(samples)-len(todo))) and (time.time()-t0)
        print(f'{done}/{len(train_bugs)} bugs sampled  ({time.time()-t0:.0f}s elapsed)')
json.dump(samples, open(CKPT, 'w'))
print('All attempts saved to', CKPT)

In [ ]:
# GRADE all attempts (CPU, subprocess + timeout) and ROUTE each bug
import subprocess, sys as _sys, tempfile
from concurrent.futures import ThreadPoolExecutor

def passes(code, bug, timeout=6):
    prog = '\n'.join(list(bug['test_imports']) + [code] + list(bug['test_list']))
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        r = subprocess.run([_sys.executable, path], capture_output=True, timeout=timeout)
        return r.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)

by_id = {b['id']: b for b in train_bugs}
routing = []
with ThreadPoolExecutor(max_workers=4) as pool:
    for n, (bid, attempts) in enumerate(samples.items()):
        bug = by_id[bid]
        flags = list(pool.map(lambda c: passes(c, bug), attempts))
        c = sum(flags)
        pile = 'sft' if c == 0 else ('easy' if c == len(flags) else 'rl')
        routing.append({'id': bid, 'category': bug['category'], 'pass_count': c,
                        'n': len(flags), 'pile': pile})
        if (n + 1) % 50 == 0:
            print(f'{n+1}/{len(samples)} graded')
json.dump(routing, open(f'{PHASE2}/routing_v0.json', 'w'), indent=1)
print('Routing saved to', f'{PHASE2}/routing_v0.json')

In [ ]:
# THE REPORT
from collections import Counter
piles = Counter(r['pile'] for r in routing)
total = len(routing)
print('=== ROUTING (Amendment A2) ===')
for p in ('sft', 'rl', 'easy'):
    print(f"  {p:<5} {piles[p]:>4}  ({piles[p]/total*100:.1f}%)")
print('\n=== RL pile = GRPO variance-gate pre-flight ===')
print(f"learnable fraction (mixed outcomes): {piles['rl']/total*100:.1f}%  (gate wants >=30%)")
print('\n=== by category (sft / rl / easy) ===')
for cat in sorted({r['category'] for r in routing}):
    sub = [r for r in routing if r['category'] == cat]
    cc = Counter(r['pile'] for r in sub)
    print(f"  {cat:<18} {cc['sft']:>3} / {cc['rl']:>3} / {cc['easy']:>3}")
print('\nmean pass rate over all attempts:',
      f"{sum(r['pass_count'] for r in routing)/sum(r['n'] for r in routing)*100:.1f}%")

## Bring back to the session
1. The **routing table** (sft / rl / easy counts and %)
2. The **learnable fraction** (the variance-gate number — ≥30% means GRPO has fuel)
3. The **per-category routing** breakdown
4. The **smoke-cell outputs** if anything looked unhealthy

After this: Phase 3 — the SFT arm (trace vs no-trace A/B, then 3 seeds).